In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
cwd = Path.cwd()
project_root = cwd.parent

In [3]:
from credit_risk.dataset import load_splits, AFTER_EDA
from credit_risk.features import prep_one_split

2026-08-13 03:26:32.948 | INFO     | credit_risk.config:<module>:11 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


In [4]:
train_df, val_df, test_df, metadata = load_splits(path=AFTER_EDA)

2026-08-13 03:26:33.713 | INFO     | credit_risk.dataset:load_splits:261 - Checking if the files exists...
2026-08-13 03:26:33.718 | INFO     | credit_risk.dataset:load_splits:263 - Loading the Cached files...
2026-08-13 03:26:34.267 | INFO     | credit_risk.dataset:load_splits:271 - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)


In [5]:
train_split, y_train = prep_one_split(df=train_df)
val_split, y_val = prep_one_split(df=val_df)

2026-08-13 03:26:34.275 | INFO     | credit_risk.features:prep_one_split:214 - Inside Function: prep_one_split
2026-08-13 03:26:34.275 | INFO     | credit_risk.features:sorting_with_issue_d:140 - Sorting the dataframe wrt to issue_d...
2026-08-13 03:26:34.478 | INFO     | credit_risk.features:sorting_with_issue_d:145 - Sorted successfully!
2026-08-13 03:26:34.478 | INFO     | credit_risk.features:split_target_and_features:150 - Inside Function: split_target_and_features
2026-08-13 03:26:34.478 | INFO     | credit_risk.features:split_target_and_features:151 - Splitting the target and the features...
2026-08-13 03:26:34.557 | INFO     | credit_risk.features:split_target_and_features:154 - features and target are splitted successfully...
2026-08-13 03:26:34.557 | INFO     | credit_risk.features:add_credit_yrs:167 - Inside Function: add_credit_yrs
2026-08-13 03:26:34.557 | INFO     | credit_risk.features:add_credit_yrs:169 - Adding the feature 'credit_yrs'
2026-08-13 03:26:34.562 | INFO   

In [6]:
pd.qcut(train_split['revol_util'], 10).value_counts()

revol_util
(23.2, 34.8]      46960
(-0.001, 23.2]    46673
(43.4, 50.8]      46664
(50.8, 57.6]      46587
(57.6, 64.3]      46575
(34.8, 43.4]      46564
(87.1, 892.3]     46494
(78.5, 87.1]      46432
(64.3, 71.1]      46411
(71.1, 78.5]      46371
Name: count, dtype: int64

In [7]:
bin_boundary = np.nanquantile(train_split['revol_util'], q=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
bin_boundary[0] = -np.inf
bin_boundary[-1] = np.inf
bin_boundary

array([-inf, 23.2, 34.8, 43.4, 50.8, 57.6, 64.3, 71.1, 78.5, 87.1,  inf])

In [8]:
train_cuts = pd.cut(train_split['revol_util'], bins=bin_boundary).value_counts()

In [9]:
val_cuts = pd.cut(val_split['revol_util'], bins=bin_boundary).value_counts()

In [10]:
train_cuts

revol_util
(23.2, 34.8]    46960
(-inf, 23.2]    46673
(43.4, 50.8]    46664
(50.8, 57.6]    46587
(57.6, 64.3]    46575
(34.8, 43.4]    46564
(87.1, inf]     46494
(78.5, 87.1]    46432
(64.3, 71.1]    46411
(71.1, 78.5]    46371
Name: count, dtype: int64

In [11]:
interval_idx = train_cuts.index.categories

In [12]:
train_cuts = train_cuts.reindex(index=interval_idx, fill_value=0)
val_cuts = val_cuts.reindex(index=interval_idx, fill_value=0)

In [13]:
train_cuts

(-inf, 23.2]    46673
(23.2, 34.8]    46960
(34.8, 43.4]    46564
(43.4, 50.8]    46664
(50.8, 57.6]    46587
(57.6, 64.3]    46575
(64.3, 71.1]    46411
(71.1, 78.5]    46371
(78.5, 87.1]    46432
(87.1, inf]     46494
Name: count, dtype: int64

In [14]:
val_cuts

(-inf, 23.2]    48209
(23.2, 34.8]    50384
(34.8, 43.4]    47263
(43.4, 50.8]    43831
(50.8, 57.6]    41627
(57.6, 64.3]    40378
(64.3, 71.1]    38082
(71.1, 78.5]    36827
(78.5, 87.1]    35184
(87.1, inf]     38258
Name: count, dtype: int64

In [15]:
train_cuts = train_cuts.clip(lower=1e-4)
val_cuts = val_cuts.clip(lower=1e-4)

In [16]:
psi_value = ((val_cuts - train_cuts) * np.log(val_cuts.divide(train_cuts))).sum()
psi_value

np.float64(10494.443587892853)

In [17]:
def psi_numeric(reference: pd.Series, target: pd.Series, bins=10) -> float:
    """Calculates the psi drift metric for a given feature

    Args:
        reference (pd.Series): reference distribution
        target (pd.Series): target distribution
        bins (int, optional): No of bins to create for both the distributions to compare and calculate psi. Defaults to 10.

    Returns:
        float: returns psi metric
    """
    
    bin_boundary = np.quantile(reference, q=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
    bin_boundary[0] = -np.inf
    bin_boundary[-1] = np.inf
    
    reference_cuts = pd.cut(reference, bins=bin_boundary).value_counts(normalize=True)
    target_cuts = pd.cut(target, bins=bin_boundary).value_counts(normalize=True)
    
    interval_idx = reference_cuts.index.categories
    
    reference_cuts = reference_cuts.reindex(index=interval_idx, fill_value=0)
    target_cuts = target_cuts.reindex(index=interval_idx, fill_value=0)
    
    reference_cuts = reference_cuts.clip(lower=1e-4)
    target_cuts = target_cuts.clip(lower=1e-4)
    
    psi_value = ((target_cuts - reference_cuts) * np.log(target_cuts.divide(reference_cuts))).sum()
    
    return psi_value.item()

In [18]:
psi_numeric(reference=train_split['dti'], target=val_split['dti'])

0.05604672920348452

In [19]:
from credit_risk.monitoring.psi import psi_numeric, psi_categorical

In [20]:
psi_numeric(reference=train_split['revol_util'], target=val_split['revol_util'])

0.013196742712624727

In [21]:
train_split['pub_rec'].value_counts()

pub_rec
0.0     404710
1.0      53027
2.0       5619
3.0       1609
4.0        520
5.0        276
6.0        136
7.0         62
8.0         29
9.0         16
10.0        13
11.0         8
13.0         2
18.0         2
12.0         2
49.0         1
54.0         1
17.0         1
34.0         1
63.0         1
21.0         1
40.0         1
15.0         1
14.0         1
16.0         1
19.0         1
Name: count, dtype: int64

### Working on Categorical features

In [22]:
train_split.select_dtypes(include='object').isna().sum(axis=0)

term                       0
emp_length             21002
home_ownership             0
verification_status        0
purpose                    0
addr_state                 0
initial_list_status        0
dtype: int64

In [23]:
reference = train_split['emp_length'][train_split['emp_length'].notna()].value_counts() / len(train_split['emp_length'])
target = val_split['emp_length'][val_split['emp_length'].notna()].value_counts() / len(val_split['emp_length'])

In [24]:
set(target.index) - set(reference.index)

set()

In [25]:
missing_ref = train_split['emp_length'].isna().sum() / len(train_split['emp_length'])
missing_tar = val_split['emp_length'].isna().sum() / len(val_split['emp_length'])

In [26]:
missing_ref

np.float64(0.045064607910874986)

In [27]:
reference['missing'] = missing_ref

In [28]:
reference

emp_length
10+ years    0.321784
2 years      0.088726
3 years      0.078495
< 1 year     0.077735
5 years      0.066018
1 year       0.063520
4 years      0.060102
7 years      0.056149
6 years      0.056006
8 years      0.048030
9 years      0.038370
missing      0.045065
Name: count, dtype: float64

In [29]:
psi_categorical(reference=train_split['emp_length'], target=val_split['emp_length'])

0.013068344024733686

In [30]:
from credit_risk.features import CATEGORICAL_COLS, NUMERICAL_COLS

In [31]:
len(CATEGORICAL_COLS) + len(NUMERICAL_COLS)

65

In [32]:
def flag_level(psi_value: float) -> str:
    if psi_value < 0.1:
        return "stable"
    elif 0.1 <= psi_value < 0.25:
        return "moderate"
    else:
        return "significant"

In [54]:
def build_drift_report(reference_df: pd.DataFrame, target_df: pd.DataFrame, target_label: str) -> pd.DataFrame:
    """Creates a complete covariate shift report using the PSI values

    Args:
        reference_df (pd.DataFrame): Reference Dataframe
        target_df (pd.DataFrame): Target Dataframe
        target_label (str): Target Dataframe Name

    Returns:
        pd.DataFrame: Drift Report Dataframe 
    """
    
    reference_df, _ = prep_one_split(df=reference_df)
    target_df, _ = prep_one_split(df=target_df)
    
    report = {
        'feature': [],
        'PSI': [],
        'drift_level': [],
        'target_label': [],
    }
    
    # Calculating PSI values for Numerical Columns
    for col in NUMERICAL_COLS:
        psi_val = psi_numeric(reference=reference_df[col], target=target_df[col])
        level = flag_level(psi_value=psi_val)
        
        report['feature'].append(col)
        report['PSI'].append(psi_val)
        report['drift_level'].append(level)
        report['target_label'].append(target_label)
        
    # Calculating PSI values for categorical Columns
    for col in CATEGORICAL_COLS:
        psi_val = psi_categorical(reference=reference_df[col], target=target_df[col])
        level = flag_level(psi_value=psi_val)
        
        report['feature'].append(col)
        report['PSI'].append(psi_val)
        report['drift_level'].append(level)
        report['target_label'].append(target_label)
        
    return pd.DataFrame(report)

In [43]:
result = build_drift_report(reference_df=train_df, target_df=val_df)

2026-08-13 03:31:32.688 | INFO     | credit_risk.features:prep_one_split:214 - Inside Function: prep_one_split
2026-08-13 03:31:32.688 | INFO     | credit_risk.features:sorting_with_issue_d:140 - Sorting the dataframe wrt to issue_d...
2026-08-13 03:31:32.820 | INFO     | credit_risk.features:sorting_with_issue_d:145 - Sorted successfully!
2026-08-13 03:31:32.820 | INFO     | credit_risk.features:split_target_and_features:150 - Inside Function: split_target_and_features
2026-08-13 03:31:32.820 | INFO     | credit_risk.features:split_target_and_features:151 - Splitting the target and the features...
2026-08-13 03:31:32.913 | INFO     | credit_risk.features:split_target_and_features:154 - features and target are splitted successfully...
2026-08-13 03:31:32.914 | INFO     | credit_risk.features:add_credit_yrs:167 - Inside Function: add_credit_yrs
2026-08-13 03:31:32.914 | INFO     | credit_risk.features:add_credit_yrs:169 - Adding the feature 'credit_yrs'
2026-08-13 03:31:32.917 | INFO   

In [44]:
len(result[result['drift_level'] == 'significant'])

33

In [48]:
result.sort_values(by='PSI', ascending=False)

,feature,PSI,drift_level
47,num_tl_op_past_12m,1.151293,significant
55,total_il_high_credit_limit,1.149529,significant
38,num_bc_tl,1.148904,significant
18,total_rev_hi_lim,1.138109,significant
48,pct_tl_nvr_dlq,1.137675,significant
...,...,...,...
60,home_ownership,0.004576,stable
12,total_acc,0.001148,stable
23,chargeoff_within_12_mths,0.001055,stable
24,delinq_amnt,0.000824,stable


In [61]:
def summarize_drift_report(report_df: pd.DataFrame) -> dict:
    """Summarizes the PSI report

    Args:
        report_df (pd.DataFrame): PSI Report
    """
    
    summary = {}
    
    summary['counts'] = pd.crosstab(report_df['target_label'], report_df['drift_level'])
    
    summary['flagged'] = report_df[report_df['drift_level'] != 'stable'].sort_values(by='PSI', ascending=False).sort_index()
    
    return summary

In [57]:
val_result = build_drift_report(reference_df=train_df, target_df=val_df, target_label='val')
test_result = build_drift_report(reference_df=train_df, target_df=test_df, target_label='test')

2026-08-13 04:18:10.304 | INFO     | credit_risk.features:prep_one_split:214 - Inside Function: prep_one_split
2026-08-13 04:18:10.304 | INFO     | credit_risk.features:sorting_with_issue_d:140 - Sorting the dataframe wrt to issue_d...
2026-08-13 04:18:10.530 | INFO     | credit_risk.features:sorting_with_issue_d:145 - Sorted successfully!
2026-08-13 04:18:10.530 | INFO     | credit_risk.features:split_target_and_features:150 - Inside Function: split_target_and_features
2026-08-13 04:18:10.530 | INFO     | credit_risk.features:split_target_and_features:151 - Splitting the target and the features...
2026-08-13 04:18:10.657 | INFO     | credit_risk.features:split_target_and_features:154 - features and target are splitted successfully...
2026-08-13 04:18:10.657 | INFO     | credit_risk.features:add_credit_yrs:167 - Inside Function: add_credit_yrs
2026-08-13 04:18:10.657 | INFO     | credit_risk.features:add_credit_yrs:169 - Adding the feature 'credit_yrs'
2026-08-13 04:18:10.664 | INFO   

In [58]:
full_report = pd.concat([val_result, test_result], ignore_index=True)

In [62]:
results = summarize_drift_report(report_df=full_report)

In [63]:
results

{'counts': drift_level   moderate  significant  stable
 target_label                               
 test                 1           33      31
 val                  1           33      31,
 'flagged':                         feature       PSI  drift_level target_label
 16                 tot_coll_amt  1.135748  significant          val
 17                  tot_cur_bal  1.133880  significant          val
 18             total_rev_hi_lim  1.138109  significant          val
 19         acc_open_past_24mths  0.783767  significant          val
 20                  avg_cur_bal  1.133214  significant          val
 ..                          ...       ...          ...          ...
 117             tot_hi_cred_lim  1.138059  significant         test
 118           total_bal_ex_mort  0.782776  significant         test
 119              total_bc_limit  0.770600  significant         test
 120  total_il_high_credit_limit  1.162756  significant         test
 129         initial_list_status  0.760